<a href="https://colab.research.google.com/github/Akshajan/IMDB-Movie-review-Sentiment-analysis/blob/main/Movie_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# step 1: Import Libraries
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pandas as pd
import numpy as np

In [ ]:
# Step 2: Load Dataset
# Loads the IMDB dataset, keeping only the top 10,000 frequent words
vocab_size = 10000
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)

In [ ]:
# --- STEP 3: PREPROCESSING ---
# Padding sequences to a uniform length of 200
max_length = 200
x_train = pad_sequences(x_train, maxlen=max_length)
x_test = pad_sequences(x_test, maxlen=max_length)

In [ ]:
# --- STEP 4: BUILD LSTM MODEL ---
# Using the architecture defined in the assignment
model = models.Sequential([
    layers.Embedding(vocab_size, 128), # input_length is now optional in newer Keras versions
    layers.LSTM(64),
    layers.Dense(1, activation='sigmoid')
])

In [ ]:
# --- STEP 5: COMPILE & TRAIN ---
# Using Adam optimizer and Binary Cross-Entropy
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Starting training for 3 epochs...")
model.fit(x_train, y_train, epochs=3, batch_size=64) # Batch size 64

Starting training for 3 epochs...
Epoch 1/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 98s 245ms/step - accuracy: 0.8129 - loss: 0.4099
Epoch 2/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 143s 248ms/step - accuracy: 0.9028 - loss: 0.2495
Epoch 3/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 96s 244ms/step - accuracy: 0.9280 - loss: 0.1905


In [ ]:
# --- STEP 6: VISUALIZE RESULTS AS A TABLE ---
# Helper to decode reviews back to text
word_index = tf.keras.datasets.imdb.get_word_index()
reverse_word_index = {v + 3: k for k, v in word_index.items()}
reverse_word_index[0], reverse_word_index[1], reverse_word_index[2] = "<PAD>", "<START>", "<UNK>"

def decode_review(text_sequence):
    return ' '.join([reverse_word_index.get(i, '?') for i in text_sequence if i > 2])

# Select first 10 samples from test set
num_samples = 10
predictions = model.predict(x_test[:num_samples])

table_data = []
for i in range(num_samples):
    review_text = decode_review(x_test[i])
    score = predictions[i][0]

    table_data.append({
        "Review Snippet": (review_text[:95] + "..."),
        "Sentiment Score": f"{score:.4f}",
        "Predicted": "Positive" if score > 0.5 else "Negative",
        "Actual": "Positive" if y_test[i] == 1 else "Negative"
    })

# Create and display the table
df = pd.DataFrame(table_data)
print("\n--- Sentiment Analysis Results ---")
display(df)

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 230ms/step

--- Sentiment Analysis Results ---


,Review Snippet,Sentiment Score,Predicted,Actual
0,please give this one a miss br br and the rest...,0.1781,Negative,Negative
1,psychological it's very interesting that rober...,0.9961,Positive,Positive
2,everyone's horror the promptly eats the mayor ...,0.7347,Positive,Positive
3,i generally love this type of movie however th...,0.7354,Positive,Negative
4,like some other people wrote i'm a die hard ma...,0.9992,Positive,Positive
5,i'm absolutely disgusted this movie isn't bein...,0.9965,Positive,Positive
6,are cleverly with long menacing shadows from a...,0.9554,Positive,Positive
7,the richard dog is to joan fontaine dog howeve...,0.0103,Negative,Negative
8,hollywood had a long love affair with bogus ni...,0.9639,Positive,Negative
9,it remains a dark film true to the way the bat...,0.9886,Positive,Positive


In [ ]:
print("\nEvaluating on test data...")
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f'\nTest Accuracy: {test_acc:.4f}')


Evaluating on test data...
782/782 ━━━━━━━━━━━━━━━━━━━━ 27s 34ms/step - accuracy: 0.8657 - loss: 0.3511

Test Accuracy: 0.8657
